# Web Scraping — Books to Scrape

This notebook scrapes [books.toscrape.com](https://books.toscrape.com), a safe practice site,
and saves all book listings into a CSV file for later preprocessing.

---

## Import Libraries

In [1]:
import requests
from bs4 import BeautifulSoup
import csv
import time

print('Libraries imported successfully!')

Libraries imported successfully!


## Define the Target URL

We start from the first catalogue page. Pagination follows the pattern `page-N.html`.

In [2]:
BASE_URL  = 'https://books.toscrape.com/catalogue/'
START_URL = 'https://books.toscrape.com/catalogue/page-1.html'

print(f'Start URL : {START_URL}')
print(f'Base URL  : {BASE_URL}')

Start URL : https://books.toscrape.com/catalogue/page-1.html
Base URL  : https://books.toscrape.com/catalogue/


## Fetch the First Page

`requests.get()` sends an HTTP GET request and returns the server response.
`BeautifulSoup` parses the raw HTML into a searchable tree.
A status code of `200` means the request was successful.

In [3]:
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/120.0.0.0 Safari/537.36'
}

response = requests.get(START_URL, headers=HEADERS)
soup = BeautifulSoup(response.text, 'lxml')

print(f'Status code : {response.status_code}')   # 200 = success

Status code : 200


## Find the Total Number of Pages

The site displays the total page count inside the pager section:

```html
<li class="current">Page 1 of 50</li>
```

We extract the number after `of` to know when to stop.

In [4]:
def get_total_pages(soup):
    pager = soup.select_one('li.current')
    if pager:
        # Text looks like "Page 1 of 50"
        return int(pager.text.strip().split()[-1])
    return 1

total_pages = get_total_pages(soup)
print(f'Total pages to scrape: {total_pages}')

Total pages to scrape: 50


## Check `robots.txt` (Ethical Scraping)

Before scraping any site, we should check whether it allows automated access.
`urllib.robotparser` reads the `robots.txt` file and tells us if our scraper is welcome.

In [5]:
from urllib.robotparser import RobotFileParser

rp = RobotFileParser()
rp.set_url('https://books.toscrape.com/robots.txt')
rp.read()

allowed = rp.can_fetch('*', START_URL)
print(f'Allowed to scrape: {allowed}')

Allowed to scrape: True


## Define the `parse_page()` Function

This function takes a parsed HTML page and extracts all book entries from it.

Each book lives inside an `<article class='product_pod'>` tag.
We loop through all of them and use CSS selectors to pull out each field:

| Field | CSS Selector / Source |
|---|---|
| Title | `h3 a` → `title` attribute |
| Price | `p.price_color` → text |
| Rating | `p.star-rating` → second CSS class |
| Availability | `p.availability` → text |
| URL | `h3 a` → `href` attribute |

In [6]:
RATING_MAP = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}

def parse_page(soup):
    entries = []

    # Each article tag = one book listing
    for article in soup.select('article.product_pod'):

        # Title — stored in the `title` attribute to preserve full text
        title_tag = article.select_one('h3 a')
        title     = title_tag['title'] if title_tag else ''

        # Relative URL to the book detail page
        book_url  = BASE_URL + title_tag['href'] if title_tag else ''

        # Price — comes as a string like '£12.99'
        price_tag = article.select_one('p.price_color')
        price     = price_tag.text.strip() if price_tag else ''

        # Rating — the word rating is stored as a CSS class
        # e.g. class="star-rating Three"  → second element = 'Three'
        rating_tag  = article.select_one('p.star-rating')
        rating_word = rating_tag['class'][1] if rating_tag else ''
        rating      = RATING_MAP.get(rating_word, 0)

        # Availability
        avail_tag = article.select_one('p.availability')
        in_stock  = avail_tag.text.strip() if avail_tag else ''

        entries.append({
            'title'       : title,
            'price'       : price,
            'rating'      : rating,
            'in_stock'    : in_stock,
            'url'         : book_url,
        })

    return entries

print('parse_page() function defined.')

# Quick sanity check on the first page
sample = parse_page(soup)
print(f'Books found on page 1: {len(sample)}')
print('First entry:', sample[0])

parse_page() function defined.
Books found on page 1: 20
First entry: {'title': 'A Light in the Attic', 'price': 'Â£51.77', 'rating': 3, 'in_stock': 'In stock', 'url': 'https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'}


## Scrape All Pages

The site uses a `Next →` link for pagination:

```html
<li class="next"><a href="page-2.html">next</a></li>
```

We follow this link until it disappears (last page). For each page we:
1. Fetch the page with `requests.get()`
2. Parse it with `BeautifulSoup`
3. Extract entries with `parse_page()`
4. Find the `Next` link and build the next URL
5. Wait 1 second before the next request (polite scraping)

In [7]:
all_entries = parse_page(soup)       # First page already fetched
print(f'Page 1/{total_pages}')

current_soup = soup
page_num     = 1

while True:
    # Find the "next" button
    next_li = current_soup.select_one('li.next a')
    if not next_li:
        break   # No more pages

    next_url     = BASE_URL + next_li['href']
    response     = requests.get(next_url, headers=HEADERS)
    current_soup = BeautifulSoup(response.text, 'lxml')
    entries      = parse_page(current_soup)
    all_entries.extend(entries)
    page_num += 1
    print(f'Page {page_num}/{total_pages}')
    time.sleep(1)   # Be polite

print()
print(f'Scraping complete! Total books collected: {len(all_entries)}')

Page 1/50
Page 2/50
Page 3/50
Page 4/50
Page 5/50
Page 6/50
Page 7/50
Page 8/50
Page 9/50
Page 10/50
Page 11/50
Page 12/50
Page 13/50
Page 14/50
Page 15/50
Page 16/50
Page 17/50
Page 18/50
Page 19/50
Page 20/50
Page 21/50
Page 22/50
Page 23/50
Page 24/50
Page 25/50
Page 26/50
Page 27/50
Page 28/50
Page 29/50
Page 30/50
Page 31/50
Page 32/50
Page 33/50
Page 34/50
Page 35/50
Page 36/50
Page 37/50
Page 38/50
Page 39/50
Page 40/50
Page 41/50
Page 42/50
Page 43/50
Page 44/50
Page 45/50
Page 46/50
Page 47/50
Page 48/50
Page 49/50
Page 50/50

Scraping complete! Total books collected: 1000


## Save to CSV

`csv.DictWriter` writes a list of dictionaries to a CSV file.
- `writeheader()` writes the column names on the first row
- `writerows()` writes all the data rows at once

In [9]:
OUTPUT_FILE = 'books_raw.csv'

with open(OUTPUT_FILE, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=all_entries[0].keys())
    writer.writeheader()
    writer.writerows(all_entries)

print(f'Saved {len(all_entries)} entries to {OUTPUT_FILE}')

Saved 1000 entries to books_raw.csv


---

The file `books_raw.csv` is ready for preprocessing with the following columns:

| Column | Description |
|---|---|
| `title` | Full book title |
| `price` | Price string including currency symbol (e.g. `£12.99`) |
| `rating` | Star rating as an integer from 1 to 5 |
| `in_stock` | Availability text (e.g. `In stock`) |
| `url` | Full URL to the book detail page |